In [ ]:
import pandas as pd
import numpy as np
import pyperf
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib.ticker as mtick

sns.set_theme(style="whitegrid")

In [ ]:
def compute_inertia(data_file, result_file, n_samples, n_features):
    """Reads the result file and calculates the sum of squared distances to centroids."""
    raw_data = np.fromfile(data_file, dtype=np.float32).reshape((n_samples, n_features))
    
    with open(result_file, 'r') as f:
        lines = f.read().splitlines()
        
    centroids = []
    clusters = {}
    
    mode = None
    for line in lines:
        if line == "[Centroids]":
            mode = "centroids"
            continue
        elif line == "[Clusters]":
            mode = "clusters"
            continue
            
        if mode == "centroids":
            centroids.append([float(x) for x in line.split()])
        elif mode == "clusters":
            parts = line.split(":")
            k = int(parts[0])
            indices_str = parts[1].strip()
            clusters[k] = [int(x) for x in indices_str.split()] if indices_str else []
            
    centroids = np.array(centroids, dtype=np.float32)
    inertia = 0.0
    
    for k, indices in clusters.items():
        if not indices:
            continue
        pts = raw_data[indices]
        c = centroids[k]
        inertia += np.sum((pts - c) ** 2)
        
    return inertia

def evaluate_parity(data_file, cpp_out_file, py_out_file, n_samples, dim):
    cpp_inertia = compute_inertia(data_file, cpp_out_file, n_samples, dim)
    py_inertia = compute_inertia(data_file, py_out_file, n_samples, dim)
    diff = abs(cpp_inertia - py_inertia) / max(cpp_inertia, py_inertia) * 100
    return cpp_inertia, py_inertia, diff

In [ ]:
manifest = pd.read_csv("benchmark_manifest.csv")
results_data = []

print("Extracting Pyperf JSONs and computing inertias...")

for index, row in manifest.iterrows():
    dim = row["Dimensions"]
    samples = row["Samples"]
    clusters = row["Clusters"]
    
    cpp_bench_base = row["CPP Bench Base"]
    py_bench_file = row["Py Bench File"]
    
    try:
        bench_cpp_aos = pyperf.BenchmarkSuite.load(f"{cpp_bench_base}_aos_to_soa.json").get_benchmarks()[0]
        bench_cpp_kmeans = pyperf.BenchmarkSuite.load(f"{cpp_bench_base}_kmeans_fit.json").get_benchmarks()[0]
        bench_py = pyperf.BenchmarkSuite.load(py_bench_file).get_benchmarks()[0]
        
        mean_cpp_aos = bench_cpp_aos.mean()
        mean_cpp_kmeans = bench_cpp_kmeans.mean()
        total_cpp = mean_cpp_aos + mean_cpp_kmeans
        
        mean_py = bench_py.mean()
        speedup = mean_py / total_cpp
        
    except Exception as e:
        print(f"Skipping {cpp_bench_base} due to missing/invalid pyperf data: {e}")
        continue

    cpp_inertia, py_inertia, diff = evaluate_parity(
        row["Data File"], row["CPP Out File"], row["Py Out File"], samples, dim
    )
    
    results_data.append({
        "Dimensions": dim,
        "Samples": samples,
        "Clusters": clusters,
        "C++ Inertia": cpp_inertia,
        "Scikit-Learn Inertia": py_inertia,
        "Inertia Diff (%)": diff,
        "C++ AoS->SoA Time (s)": mean_cpp_aos,
        "C++ Math Time (s)": mean_cpp_kmeans,
        "C++ Total (s)": total_cpp,
        "Python Total (s)": mean_py,
        "Speedup (x)": speedup
    })

df = pd.DataFrame(results_data)
print("Data loading complete!")

display(df.head(10))

In [ ]:
max_diff = df["Inertia Diff (%)"].max()
mean_diff = df["Inertia Diff (%)"].mean()

print(f"Maximum Inertia Difference: {max_diff:.6f}%")
print(f"Average Inertia Difference: {mean_diff:.6f}%\n")

parity_summary = df[[
    "Dimensions", 
    "Samples", 
    "Clusters", 
    "C++ Inertia", 
    "Scikit-Learn Inertia", 
    "Inertia Diff (%)"
]]

parity_summary = parity_summary.sort_values(by="Inertia Diff (%)", ascending=False)

print("Top 10 configurations with the highest inertia difference:")
display(parity_summary.head(10))

In [ ]:
largest_samples = df["Samples"].max()
df_memory = df[df["Samples"] == largest_samples].groupby("Dimensions")[
    ["C++ AoS->SoA Time (s)", "C++ Math Time (s)"]
].mean()

df_memory_pct = df_memory.div(df_memory.sum(axis=1), axis=0) * 100

ax = df_memory_pct.plot(
    kind='bar', 
    stacked=True, 
    figsize=(10, 6), 
    color=["#e74c3c", "#3498db"]
)

plt.title(f"Normalized C++ Execution Breakdown: Memory Tax vs. Math\n(Fixed at {largest_samples:,} Samples)")
plt.xlabel("Dimensions")
plt.ylabel("Percentage of Total Execution Time (%)")
plt.xticks(rotation=0)

plt.legend(
    ["AoS to SoA Conversion", "K-Means Math (SIMD)"], 
    title="Operation", 
    bbox_to_anchor=(1.05, 1), 
    loc='upper left'
)

ax.yaxis.set_major_formatter(mtick.PercentFormatter())

plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()

In [ ]:
heatmap_data = df.groupby(["Dimensions", "Samples"])["Speedup (x)"].mean().reset_index()
heatmap_pivot = heatmap_data.pivot(index="Dimensions", columns="Samples", values="Speedup (x)")

plt.figure(figsize=(10, 6))
sns.heatmap(
    heatmap_pivot, 
    annot=True,
    fmt=".2f", 
    cmap="magma",     
    cbar_kws={'label': 'Speedup Multiplier (x)'},
    linewidths=.5
)

plt.title("Overall Speedup Heatmap (C++ vs. Scikit-Learn)")
plt.xlabel("Number of Samples")
plt.ylabel("Dimensions")
plt.tight_layout()
plt.show()

In [ ]:
base_k = df["Clusters"].min()

baseline_df = df[df["Clusters"] == base_k][["Dimensions", "Samples", "Speedup (x)"]].copy()
baseline_df.rename(columns={"Speedup (x)": "Base_Speedup"}, inplace=True)
df_norm = pd.merge(df, baseline_df, on=["Dimensions", "Samples"])

df_norm["Performance Retention (%)"] = (df_norm["Speedup (x)"] / df_norm["Base_Speedup"]) * 100

g = sns.relplot(
    data=df_norm,
    x="Clusters",
    y="Performance Retention (%)",
    hue="Samples",   
    col="Dimensions",
    col_wrap=2,      
    kind="line",
    marker="o",
    palette="viridis",
    height=4,  
    aspect=1.2,
    facet_kws={'sharey': True}
)

g.fig.suptitle(f"Impact of Cluster Count on Speedup Efficiency by Dimension\n(Normalized against K={base_k})", y=1.05)
g.set_axis_labels("Number of Clusters (K)", "Performance Retention (%)")

for ax in g.axes.flat:
    ax.axhline(100, color='red', linestyle='--', alpha=0.5)
    ax.grid(axis='y', linestyle='--', alpha=0.7)
    
    ax.xaxis.set_major_locator(mtick.MaxNLocator(integer=True))

plt.show()